# Hyperparameter Tuning — FashionMNIST Clean Solution

Grid Search, Random Search, and Optuna Bayesian optimization on a simple FashionMNIST classifier.

**Key concepts:** GridSearch, RandomSearch, Optuna, learning rate & batch size sensitivity.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import optuna
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1–2: Dataset & DataLoaders

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_subset, _ = random_split(full_train, [10000, len(full_train)-10000])
test_subset, _ = random_split(full_test, [2000, len(full_test)-2000])

def get_loader(subset, batch_size):
    return DataLoader(subset, batch_size=batch_size, shuffle=True)

print(f"Train: {len(train_subset)}, Test: {len(test_subset)}")

## Step 3: Base Model

In [ ]:
class FashionNN(nn.Module):
    def __init__(self, hidden1=128, hidden2=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 10)
        )
    def forward(self, x):
        return self.network(x)

def quick_train(model, loader, optimizer, epochs=5):
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
    model.eval()
    total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            total += loss_fn(model(images), labels).item()
    return total / len(loader)

## Grid Search

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [32, 64]
best_loss, best_params = float('inf'), {}

for lr in learning_rates:
    for bs in batch_sizes:
        loader = get_loader(train_subset, bs)
        model = FashionNN().to(device)
        opt = optim.SGD(model.parameters(), lr=lr)
        loss = quick_train(model, loader, opt, epochs=3)
        print(f"LR: {lr}, BS: {bs}, Loss: {loss:.4f}")
        if loss < best_loss:
            best_loss, best_params = loss, {'lr': lr, 'batch_size': bs}

print(f"Best Grid Search: {best_params}")

## Random Search

In [ ]:
lrs = [0.0001, 0.001, 0.01, 0.1]
bss = [16, 32, 64, 128]
best_loss_rs, best_params_rs = float('inf'), {}

for _ in range(5):
    lr = random.choice(lrs)
    bs = random.choice(bss)
    loader = get_loader(train_subset, bs)
    model = FashionNN().to(device)
    opt = optim.SGD(model.parameters(), lr=lr)
    loss = quick_train(model, loader, opt, epochs=3)
    print(f"LR: {lr}, BS: {bs}, Loss: {loss:.4f}")
    if loss < best_loss_rs:
        best_loss_rs, best_params_rs = loss, {'lr': lr, 'batch_size': bs}

print(f"Best Random Search: {best_params_rs}")

## Optuna Bayesian Optimization

In [ ]:
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    neurons = trial.suggest_int('neurons', 32, 256)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, neurons),
        nn.ReLU(),
        nn.Linear(neurons, 10)
    ).to(device)
    opt = optim.SGD(model.parameters(), lr=lr)
    loader = get_loader(train_subset, 32)
    return quick_train(model, loader, opt, epochs=3)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8, show_progress_bar=False)
print(f"Best Optuna: {study.best_params}")